# Task 3 — Official scoring of saved development responses
CPU-only scoring of four saved 69-case static development runs. The pinned RobustBench-TC loader and parser reconstruct each source-file-qualified sample; the released deterministic scorer evaluates the saved prediction against that sample's reference answers.
All 276 attempts remain in the official scoring denominator, including truncations and invalid token evidence. Official correctness is preserved independently of response status. This diagnostic does not introduce a deployment acceptance rule, modify parser/generation settings or fit routing thresholds.
Whole-output uncertainty values are joined from the audited records only where eligible. Original records remain unchanged. Output: `MyDrive/UQRoute-TC/throughput/four-model-static-strata-scoring-v1/official_scoring.json`.
Runtime: Colab CPU. Execute setup and scoring cells in order.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, math, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/UQRoute-TC/throughput')
audit_path = root/'four-model-static-strata-audit-v1/record_audit.json'
sha = lambda b: hashlib.sha256(b).hexdigest()
assert sha(audit_path.read_bytes()) == 'e5f505b102695ff95db98a13b7d143f0570909a8a455e0021fbd8d74f8375c7f', 'Saved audit changed; review its provenance before scoring'
audit = json.loads(audit_path.read_text())
assert audit['total_records']==276
BENCHMARK_REVISION = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
benchmark = Path('/content/robustbench-tc-static-scoring')
if not benchmark.exists():
    subprocess.run(['git','clone','--no-checkout','https://github.com/WillChow66/robustbench-tc-release.git',str(benchmark)],check=True)
    subprocess.run(['git','-C',str(benchmark),'checkout','--detach',BENCHMARK_REVISION],check=True)
assert subprocess.check_output(['git','-C',str(benchmark),'rev-parse','HEAD'],text=True).strip()==BENCHMARK_REVISION
assert not subprocess.check_output(['git','-C',str(benchmark),'status','--porcelain'],text=True).strip()
runner_path=benchmark/'scripts/run_eval.py'
scorer_path=benchmark/'leaderboard/scoring/eval_results.py'
assert sha(runner_path.read_bytes())=='e305a816c96ed574ad18ca90871775721fd79fcf3533a60077b26d1fdeef20d8'
assert sha(scorer_path.read_bytes())=='2e29741b3fe7c46d5cc99da1a0a6c75ff52d9d8c4d6156db21cdc55ef115a894'
# Dependencies support the official runner import; no inference client is instantiated.
subprocess.run([sys.executable,'-m','pip','install','-q','openai==2.54.0','tqdm'],check=True)
def load_module(name,path):
    spec=importlib.util.spec_from_file_location(name,path)
    module=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
runner=load_module('pinned_static_runner',runner_path)
scorer=load_module('pinned_static_scorer',scorer_path)
print('Pinned loader, parser and deterministic scorer loaded on CPU.')


In [ ]:
def evaluate_saved_sample(sample, record):
    assert str(sample['id'])==record['id'] and sample['benchmark']==record['benchmark']
    assert 'multi_turn' not in str(sample.get('category') or '')
    prediction=record['prediction']
    parsed=runner.parse_tool_calls(prediction['raw_output'],sample['benchmark'])
    canonical=lambda value:json.dumps(value,sort_keys=True,ensure_ascii=False)
    assert canonical(parsed)==canonical(prediction['tool_calls']), 'Pinned parser replay differs from saved calls'
    scored_input=dict(sample)
    scored_input['prediction']=prediction
    result=scorer.score_record(scored_input)
    assert isinstance(result['correct'],bool)
    return result

models={}
source_samples={}
for model_key,model_audit in audit['models'].items():
    folder=root/(model_key+'-static-strata-v1')
    manifest_path=folder/'manifest.json'
    assert sha(manifest_path.read_bytes())==model_audit['manifest_sha256']
    manifest=json.loads(manifest_path.read_text())
    assert manifest['benchmark_revision']==BENCHMARK_REVISION
    assert manifest['official_runner_sha256']==sha(runner_path.read_bytes())
    expected={r['case_key']:r for r in model_audit['rows']}
    assert set(expected)==set(manifest['case_keys']) and len(expected)==69
    for file_name,file_hash in manifest['source_hashes'].items():
        data_path=benchmark/'hf_data/datasets/api_eval'/file_name
        assert sha(data_path.read_bytes())==file_hash
        if file_name not in source_samples:
            loaded=runner.load_samples(data_path)
            wanted={key.split(':',1)[1] for key in expected if key.split(':',1)[0]==file_name}
            selected=[s for s in loaded if str(s['id']) in wanted]
            indexed={str(s['id']):s for s in selected}
            assert len(indexed)==len(selected)==len(wanted)
            source_samples[file_name]=indexed
    paths=sorted((folder/'cases').glob('*/attempts/*.json'))
    assert len(paths)==69
    rows=[]
    aggregate_inputs=[]
    seen=set()
    for path in paths:
        record=json.loads(path.read_text())
        key=record['case_key'];assert key not in seen;seen.add(key)
        audited=expected[key]
        assert sha(path.read_bytes())==audited['record_sha256'], 'Saved response changed after audit'
        assert record['status']==audited['status']
        assert key==record['source_file']+':'+record['id']
        sample=source_samples[record['source_file']][record['id']]
        result=evaluate_saved_sample(sample,record)
        rows.append({k:audited[k] for k in ['case_key','base_id','benchmark','source_file','status','finish_reason','record_sha256','usage','elapsed_seconds','whole_output_score_eligible','whole_output_scores']} | {'official_score':result,'category':sample.get('category'),'perturbation':sample.get('perturbation')})
        aggregate_inputs.append({'id':record['id'],'case_key':key,'benchmark':sample['benchmark'],'perturbation':sample.get('perturbation'),'score':result,'status':record['status']})
    assert seen==set(expected)
    aggregate=scorer.aggregate(aggregate_inputs)
    assert aggregate['overall']['n']==69
    by_status={status:scorer.aggregate([r for r in aggregate_inputs if r['status']==status])['overall'] for status in model_audit['status_counts']}
    models[model_key]={'model_id':model_audit['model_id'],'official_aggregate':aggregate,'official_by_response_status':by_status,'rows':rows,'fresh_generation_sessions':model_audit['fresh_generation_sessions']}
    print(model_key, json.dumps({'official_correct':aggregate['overall']['correct'],'scored':69,'by_response_status':by_status}))
report={'scope':'Descriptive official scoring of a coverage-stratified development diagnostic; no population-weighted, held-out or routing-benefit estimate','benchmark_revision':BENCHMARK_REVISION,'official_runner_sha256':sha(runner_path.read_bytes()),'official_scorer_sha256':sha(scorer_path.read_bytes()),'source_record_audit_sha256':sha(audit_path.read_bytes()),'scored_model_case_pairs':sum(len(m['rows']) for m in models.values()),'models':models,'policy':'Official scorer correctness is retained for all saved responses, independently of status. Unavailable uncertainty scores remain null. No deployment acceptance or routing policy is selected.','limitations':['Only 69 coverage-stratified development cases per model; rows are not independent because related variants share base tasks.','Truncated or evidence-invalid outputs may still obtain an official parser/scorer match; report their statuses separately.','Released scorer semantics, including its benchmark-specific tolerances, are preserved.','Thresholds, calibration, routing and held-out evaluation remain pending.']}
assert report['scored_model_case_pairs']==276
destination=root/'four-model-static-strata-scoring-v1'
destination.mkdir(exist_ok=True)
output=destination/'official_scoring.json'
output.write_text(json.dumps(report,indent=2)+'\n')
print('Scored model–case pairs:',report['scored_model_case_pairs'])
print('Official scoring report:',output)
from google.colab import files
files.download(str(output))
